# Everyone's Combined Work: Team Data Quality Report

**Milestone 1, Task 1 | Pharma 1A (WebMD) | Break Through Tech AI Studio**

This notebook brings every team member's data quality report together in one place and checks each finding against Rhea's full report (`01_data_quality_report.ipynb` and `reports/data_quality_issues.csv`, issues DQ-01 to DQ-35).

| Team member | Columns covered | Branch | Notebook |
|---|---|---|---|
| Rhea | brand, sub_product_name, country, pillar (plus the full 35-issue report) | main (fork) | `01_data_quality_report.ipynb`, `02_data_cleaning.ipynb` |
| Emma | media_type, profession, specialty, campaign_start_date, campaign_end_date | `emma-data-quality-report` | `data_quality_report_variables_E-H.ipynb` |
| Abhiram | unique_reach, unique_engaged_hcps, total_engagements, total_budget_spent, engagement_rate | `abhiramsegu-data-quality-report-j-n` | `data_quality_report_variables_J-N.ipynb` |
| Bakari | engagement_frequency, pre_NRx, post_NRx, pre_avg_NRx | `8-task-1-data-quality-report---bakari-kerr` | `Milestone1.ipynb` |
| Timothy | post_avg_NRx, pre_NRx_avg, npi_specialty, region, NRx_lift | `7-task-1---data-quality-report---timothy-hsu` | `TimothyHsu_Data_Quality_Report.ipynb` |
| Nida | NRx_lift_pct, spend_per_NRx_lift | `9-task-1---nida-syed` | `BTT.ipynb` |

**How to read the comparison**

- **Agrees:** same issue and same fix.
- **Agrees, different fix:** same issue found, but the proposed fix is different. The team should pick one.
- **Adds detail:** the teammate found something Rhea's report does not mention.
- **Only in Rhea's report:** an issue in that teammate's columns that their notebook does not cover yet.

## Setup

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

df = pd.read_excel('../data/BreakThroughTech_training_data_1.xlsx')
rhea = pd.read_csv('../reports/data_quality_issues.csv')
n = len(df)
print(f'Raw data: {df.shape[0]} rows x {df.shape[1]} columns')
print(f"Rhea's report: {len(rhea)} issues ({rhea.issue_id.iloc[0]} to {rhea.issue_id.iloc[-1]})")

Raw data: 417 rows x 28 columns
Rhea's report: 35 issues (DQ-01 to DQ-35)


In [2]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())

def iqr_high(s):
    q1, q3 = s.quantile([0.25, 0.75])
    return int((s > q3 + 1.5 * (q3 - q1)).sum())

def rhea_issues(*ids):
    return rhea.loc[rhea.issue_id.isin(ids), ['issue_id', 'columns', 'issue', 'rows_affected', 'proposed_fix']]

---
## 1. Rhea: brand, sub_product_name, country, pillar

No teammate notebook covers these columns, so they come straight from Rhea's report. Emma's finding that 56 rows are missing targeting fields together confirms DQ-03 from a second angle.

In [3]:
rhea_issues('DQ-01', 'DQ-02', 'DQ-03', 'DQ-04', 'DQ-05', 'DQ-06')

,issue_id,columns,issue,rows_affected,proposed_fix
0,DQ-01,brand,"Inconsistent naming: 73 raw values mix spelling variants of the same company, company+drug compounds and channel tag...",183,"Map every raw value to three clean columns: company (standardized name), product (drug name if present) and channel_..."
1,DQ-02,sub_product_name,"No nulls and consistent casing, but two categories are singletons (CONDITION MESSAGE, DIRECT MAIL), which cannot be ...",2,Keep values as is. Group categories with fewer than 5 rows into 'OTHER' only in the encoded modeling file.
2,DQ-03,"country, pillar",Part of a 56-row block where country and pillar are missing together with Emma's targeting fields (mostly EMAIL ALER...,56,Fill with an explicit 'unknown' category and add a targeting_info_missing flag so models can learn from the pattern.
3,DQ-04,country,"56 nulls (13.4%). Values are lowercase ISO-3 codes (usa, zaf, vgb, vir...) with no readable names.",56,"Fill nulls with 'UNKNOWN', convert codes to uppercase ISO-3 and add a country_name column."
4,DQ-05,country,"Sparse categories: vnm and ind (3 rows each), zmb (4) and zwe (5) are too small to model on their own.",15,Group countries with fewer than 10 rows into 'OTHER' in the encoded file.
5,DQ-06,pillar,56 nulls (same rows as the missing country block). 'other' has only 2 rows.,58,Fill nulls with 'unknown'. Keep 'other' as is and let it fall into the rare bucket during encoding.


In [4]:
block = df[['country', 'pillar', 'media_type', 'profession', 'specialty']].isna().all(axis=1)
print('Rows missing country, pillar, media_type, profession and specialty together:', block.sum())
print('Raw brand values:', df['brand'].nunique())

Rows missing country, pillar, media_type, profession and specialty together: 56
Raw brand values: 73


---
## 2. Emma: media_type, profession, specialty, campaign_start_date, campaign_end_date

**Emma's findings (summary)**

- `media_type`: 80 nulls (19.2%). 56 of them sit in the block that is also missing profession, specialty, country and pillar. Fix: recode to UNKNOWN, and explore dropping the block.
- `profession`: 56 nulls. Mixed representation, 296 numeric codes and 65 text labels, with no codebook. Fix: request a data dictionary, treat codes as their own categories, cast to string.
- `specialty`: 295 'undefined' plus 57 nulls plus 11 numeric codes, so only 54 rows (12.9%) are usable. Fix: recode 'undefined' and codes to NaN, use `npi_specialty` instead, remove `specialty` from the model.
- `campaign_start_date`: 3 nulls, stored as text. Fix: drop the 3 rows and convert to datetime.
- `campaign_end_date`: 414 rows are the 2099-12-31 placeholder and 3 are null. Fix: drop the column.

In [5]:
prof = df['profession'].dropna()
checks = {
    'media_type nulls': df['media_type'].isna().sum(),
    'media_type nulls outside the 56-row block': (df['media_type'].isna() & ~block).sum(),
    'profession nulls': df['profession'].isna().sum(),
    'profession numeric codes': prof.apply(lambda x: isinstance(x, (int, np.integer))).sum(),
    'profession text labels': prof.apply(lambda x: isinstance(x, str)).sum(),
    "specialty 'undefined'": (df['specialty'] == 'undefined').sum(),
    'specialty nulls': df['specialty'].isna().sum(),
    'specialty usable text (urology, vascular medicine)': df['specialty'].isin(['urology', 'vascular medicine']).sum(),
    'campaign_start_date nulls': df['campaign_start_date'].isna().sum(),
    'campaign_end_date = 2099 placeholder': (df['campaign_end_date'] == '2099-12-31 23:59:59.999999 UTC').sum(),
    'psychologist + urology rows': ((df['profession'] == 'psychologist') & (df['specialty'] == 'urology')).sum(),
}
pd.Series(checks, name='count').to_frame()

,count
media_type nulls,80
media_type nulls outside the 56-row block,24
profession nulls,56
profession numeric codes,296
profession text labels,65
specialty 'undefined',295
specialty nulls,57
"specialty usable text (urology, vascular medicine)",54
campaign_start_date nulls,3
campaign_end_date = 2099 placeholder,414


**Check against Rhea's report:** every count above matches. Two things to settle as a team:

1. **Missing start dates.** Emma drops the 3 rows. Rhea imputes them with the company's median start date and adds a flag. The cell below shows these are the same 3 rows where `total_spend` is blank (DQ-32), which Rhea already leaves out of spend training. Either way, those rows stay out of the model, so the choice mostly matters for EDA.
2. **specialty.** Emma removes it from the model. Rhea keeps it as 'unknown' with a `has_specialty` flag and uses `npi_specialty` as the main feature. Both agree `npi_specialty` should lead.

Rhea's report also flags 36 rows pairing psychologist with urology (DQ-11), which is not in Emma's notebook yet.

In [6]:
no_start = df.index[df['campaign_start_date'].isna()].tolist()
print('Rows with no start date:', no_start)
print('total_spend on those rows:', df.loc[no_start, 'total_spend'].tolist(), '(blank in the raw file)')

Rows with no start date: [170, 183, 282]
total_spend on those rows: [nan, nan, nan] (blank in the raw file)


---
## 3. Abhiram: unique_reach, unique_engaged_hcps, total_engagements, total_budget_spent, engagement_rate

**Abhiram's findings (summary)**

- No nulls, correct numeric types and no duplicate rows in these five columns.
- `total_budget_spent`: 2 rows at $0 (JANSSEN BIOTECH and LILLY) even though they have hundreds of engagements. Fix: review, and do not impute until confirmed.
- `engagement_rate`: derived from unique_engaged_hcps / unique_reach and rounded to 4 decimals. Fix: recompute or document the rounding.
- Strong right skew in reach, engaged HCPs, engagements and budget. Fix: consider log1p for linear models.

In [7]:
cols = ['unique_reach', 'unique_engaged_hcps', 'total_engagements', 'total_budget_spent', 'engagement_rate']
recalc_rate = (df['unique_engaged_hcps'] / df['unique_reach']).round(4)
checks = {
    'nulls across the 5 columns': int(df[cols].isna().sum().sum()),
    'total_budget_spent = 0': (df['total_budget_spent'] == 0).sum(),
    'engagement_rate mismatches after rounding to 4 decimals': (recalc_rate != df['engagement_rate']).sum(),
}
print(pd.Series(checks, name='count').to_frame(), '\n')
print('Skew:')
print(df[cols[:4]].skew().round(2))
print('\nBrands with $0 budget:', df.loc[df['total_budget_spent'] == 0, 'brand'].tolist())

                                                         count
nulls across the 5 columns                                   0
total_budget_spent = 0                                       2
engagement_rate mismatches after rounding to 4 decimals      0 

Skew:
unique_reach           3.27
unique_engaged_hcps    4.23
total_engagements      4.02
total_budget_spent     5.77
dtype: float64

Brands with $0 budget: ['JANSSEN BIOTECH', 'LILLY']


**Check against Rhea's report:** matches DQ-16 (skew and log1p), DQ-17 (total_engagements has no nulls) and DQ-20 (engagement_rate). Recomputing engagement_rate gives 0 mismatches at 4 decimals, so documenting the rounding is enough and no recompute is needed.

On budget, Rhea's DQ-18 goes a step further: besides the 2 rows at $0, 5 more rows have budgets far below cpe x engagements, for 7 rows in total, all marked with a `budget_anomaly` flag.

Not in Abhiram's notebook yet: DQ-19, the leakage risk for the CPE target (cpe = total_budget_spent / total_engagements).

---
## 4. Bakari: engagement_frequency, pre_NRx, post_NRx, pre_avg_NRx

**Bakari's findings (summary)**

- All four columns are numeric with no nulls. No changes needed other than normalization.

In [8]:
cols = ['engagement_frequency', 'pre_NRx', 'post_NRx', 'pre_avg_NRx']
print('Nulls:', int(df[cols].isna().sum().sum()))
df[cols].describe().round(2)

Nulls: 0


,engagement_frequency,pre_NRx,post_NRx,pre_avg_NRx
count,417.00,417.00,417.00,417.00
mean,2.92,1331.05,1610.64,88.83
std,0.93,1077.08,1310.73,85.16
min,1.20,35.08,41.91,2.23
25%,2.19,541.09,631.51,30.44
50%,2.96,972.62,1167.58,60.95
75%,3.69,1988.51,2351.16,125.19
max,4.50,3975.73,5138.22,487.41


**Check against Rhea's report:** agrees with DQ-21 (no cleaning needed). Normalization is a modeling step (scaling features before training) rather than a cleaning fix, so it can go into the modeling pipeline.

Not in Bakari's notebook yet: DQ-22. `post_NRx` is measured after the campaign, so it is not known when budgets are planned and should stay out of the feature sets for total_spend and CPE.

---
## 5. Timothy: post_avg_NRx, pre_NRx_avg, npi_specialty, region, NRx_lift

**Timothy's findings (summary)**

- `post_avg_NRx`: 20 unusually high values. Fix: cap at the IQR upper limit.
- `pre_NRx_avg`: 14 unusually high values. Fix: cap at the IQR upper limit.
- `npi_specialty`: 1 coded value, '3320 - Anesthesiologists'. Fix: rename to 'Anesthesiology'.
- `region`: 'West South Central' appears only once. Fix: group into 'Southwest'.
- `NRx_lift`: 106 one-cent rounding differences against post_NRx minus pre_NRx. Fix: recalculate.

In [9]:
recalc_lift = (df['post_NRx'] - df['pre_NRx']).round(2)
checks = {
    'post_avg_NRx high outliers (IQR)': iqr_high(df['post_avg_NRx']),
    'pre_NRx_avg high outliers (IQR)': iqr_high(df['pre_NRx_avg']),
    "npi_specialty = '3320 - Anesthesiologists'": (df['npi_specialty'] == '3320 - Anesthesiologists').sum(),
    "region = 'West South Central'": (df['region'] == 'West South Central').sum(),
    'NRx_lift rows off from post - pre': (df['NRx_lift'] != recalc_lift).sum(),
}
print(pd.Series(checks, name='count').to_frame(), '\n')
print('Both one-off labels are on the same row:')
print(df.loc[df['npi_specialty'] == '3320 - Anesthesiologists', ['brand', 'npi_specialty', 'region', 'country']])

                                            count
post_avg_NRx high outliers (IQR)               20
pre_NRx_avg high outliers (IQR)                14
npi_specialty = '3320 - Anesthesiologists'      1
region = 'West South Central'                   1
NRx_lift rows off from post - pre             106 

Both one-off labels are on the same row:
                        brand             npi_specialty              region country
136  INTRA-CELLULAR THERAPIES  3320 - Anesthesiologists  West South Central     NaN


In [10]:
# What happens to NRx_lift_pct if NRx_lift is recalculated?
pct_now = df['NRx_lift'] / df['pre_NRx'] * 100
pct_new = recalc_lift / df['pre_NRx'] * 100
print('NRx_lift_pct rows off by more than 0.01 points, using the current NRx_lift:', ((pct_now - df['NRx_lift_pct']).abs() > 0.01).sum())
print('NRx_lift_pct rows off by more than 0.01 points, using the recalculated NRx_lift:', ((pct_new - df['NRx_lift_pct']).abs() > 0.01).sum())

NRx_lift_pct rows off by more than 0.01 points, using the current NRx_lift: 0
NRx_lift_pct rows off by more than 0.01 points, using the recalculated NRx_lift: 4


**Check against Rhea's report:** all counts match, and the npi_specialty and region fixes are identical (DQ-25, DQ-26). Things to settle:

1. **Capping outliers.** Rhea keeps the real values. `post_avg_NRx` is a post-campaign column (DQ-23), so it stays out of the model either way and capping only changes EDA charts. For `pre_NRx_avg`, Rhea's bigger concern is that its definition is undocumented and differs from `pre_avg_NRx` (DQ-24). Worth asking WebMD before capping.
2. **NRx_lift.** Rhea keeps the provided values (DQ-28). The cell above shows that recalculating NRx_lift pulls 4 rows of `NRx_lift_pct` out of line, so if the team recalculates, NRx_lift_pct should be recalculated with it.

Not in Timothy's notebook yet: DQ-27, where US regions are assigned to non-US countries on 160 rows.

---
## 6. Nida: NRx_lift_pct, spend_per_NRx_lift

**Nida's findings (summary)**

- No nulls in either column. No NRx_lift_pct values outside a sensible range.
- `spend_per_NRx_lift`: 0 infinite, 0 negative and 2 zero values.
- IQR outliers: 8 in NRx_lift_pct and 49 in spend_per_NRx_lift. No duplicate rows.
- No proposed fixes are written in the notebook yet.

In [11]:
checks = {
    'nulls in the 2 columns': int(df[['NRx_lift_pct', 'spend_per_NRx_lift']].isna().sum().sum()),
    'spend_per_NRx_lift = 0': (df['spend_per_NRx_lift'] == 0).sum(),
    'NRx_lift_pct outliers (IQR)': iqr_outliers(df['NRx_lift_pct']),
    'spend_per_NRx_lift outliers (IQR)': iqr_outliers(df['spend_per_NRx_lift']),
    'duplicate rows': df.duplicated().sum(),
}
pd.Series(checks, name='count').to_frame()

,count
nulls in the 2 columns,0
spend_per_NRx_lift = 0,2
NRx_lift_pct outliers (IQR),8
spend_per_NRx_lift outliers (IQR),49
duplicate rows,0


**Check against Rhea's report:** agrees with DQ-29 and DQ-35. Nida adds the 8 NRx_lift_pct outliers, which Rhea's report does not mention. Rhea adds that `spend_per_NRx_lift` rebuilds total_spend (total_spend / NRx_lift), so it must never be a feature for the spend target.

Not in Nida's notebook yet: the cpe and total_spend issues in DQ-30 to DQ-34 (93 missing cpe values, whether cpe is the right target, 3 median-filled spend values, 8 near-zero spend rows and the extreme range of total_spend).

---
## 7. Master comparison table

Every teammate finding lined up with the matching issue in Rhea's report. This is also saved as `reports/team_combined_comparison.csv`.

In [12]:
rows = [
    # member, column, teammate finding, Rhea issue, status, note
    ("Emma", "media_type", "80 nulls, 56 tied to the missing-metadata block", "DQ-07, DQ-08", "Agrees", "Emma also suggests exploring dropping the block. Rhea keeps the rows with an 'unknown' category and a targeting_info_missing flag."),
    ("Emma", "profession", "56 nulls", "DQ-10", "Agrees", ""),
    ("Emma", "profession", "Numeric codes mixed with text, no codebook", "DQ-09", "Agrees", "Both treat codes as categories and ask WebMD for the dictionary."),
    ("Emma", "specialty", "'undefined' and nulls hide missing data, use npi_specialty", "DQ-12", "Agrees, different fix", "Emma removes specialty from the model. Rhea keeps it as 'unknown' plus a has_specialty flag."),
    ("Emma", "specialty", "11 numeric codes", "DQ-13", "Agrees, different fix", "Emma recodes them to NaN. Rhea keeps them as code_ labels."),
    ("Emma", "campaign_start_date", "3 nulls, stored as text", "DQ-14", "Agrees, different fix", "Emma drops the 3 rows. Rhea imputes the company median date and flags them. Same 3 rows where total_spend is blank (DQ-32)."),
    ("Emma", "campaign_end_date", "2099 placeholder on 414 rows, drop column", "DQ-15", "Agrees", ""),
    ("Emma", "profession, specialty", "(not covered)", "DQ-11", "Only in Rhea's report", "36 rows pair psychologist with urology."),
    ("Abhiram", "total_budget_spent", "2 rows at $0 with real engagement", "DQ-18", "Agrees", "Rhea also flags 5 more rows with budgets far below cpe x engagements (7 in total)."),
    ("Abhiram", "engagement_rate", "Derived and rounded to 4 decimals", "DQ-20", "Agrees", "0 mismatches after rounding, so documenting the rounding is enough."),
    ("Abhiram", "reach, engaged HCPs, engagements, budget", "Strong right skew, consider log1p", "DQ-16", "Agrees", ""),
    ("Abhiram", "total_engagements", "No nulls", "DQ-17", "Agrees", "The milestone plan expected this column to be empty. Confirm the file version with the coach."),
    ("Abhiram", "total_budget_spent, total_engagements", "(not covered)", "DQ-19", "Only in Rhea's report", "Leakage for the CPE target: budget / engagements = cpe."),
    ("Bakari", "engagement_frequency, pre_NRx, post_NRx, pre_avg_NRx", "No nulls, all numeric, normalize", "DQ-21", "Agrees", "Normalization belongs in the modeling pipeline, not cleaning."),
    ("Bakari", "post_NRx", "(not covered)", "DQ-22", "Only in Rhea's report", "Post-campaign value, keep out of the feature sets."),
    ("Timothy", "post_avg_NRx", "20 high outliers, cap at IQR limit", "DQ-23", "Agrees, different fix", "Rhea keeps real values. Post-campaign, so it is excluded from features either way."),
    ("Timothy", "pre_NRx_avg", "14 high outliers, cap at IQR limit", "DQ-24", "Agrees, different fix", "Rhea flags an undocumented definition that differs from pre_avg_NRx. Ask WebMD before capping."),
    ("Timothy", "npi_specialty", "'3320 - Anesthesiologists', rename to 'Anesthesiology'", "DQ-25", "Agrees", "Same fix."),
    ("Timothy", "region", "'West South Central' singleton, map to 'Southwest'", "DQ-26", "Agrees", "Same fix. Same row (136) as the npi_specialty fix."),
    ("Timothy", "NRx_lift", "106 one-cent differences, recalculate", "DQ-28", "Agrees, different fix", "Rhea keeps the provided values. Recalculating puts 4 rows of NRx_lift_pct out of line."),
    ("Timothy", "region", "(not covered)", "DQ-27", "Only in Rhea's report", "US regions on non-US countries (160 rows)."),
    ("Nida", "NRx_lift_pct", "No nulls or out-of-range values, 8 IQR outliers", "DQ-29", "Adds detail", "Rhea's report does not mention the 8 outliers."),
    ("Nida", "spend_per_NRx_lift", "2 zeros, 49 IQR outliers", "DQ-35", "Agrees", "Rhea adds that this column leaks total_spend."),
    ("Nida", "cpe, total_spend, actual_spend_per_engagement", "(not covered)", "DQ-30 to DQ-34", "Only in Rhea's report", "cpe nulls and target choice, median-filled and near-zero spend, spend range."),
    ("Rhea", "brand, sub_product_name, country, pillar", "Rhea's assigned columns", "DQ-01 to DQ-06", "Rhea only", "No teammate notebook covers these columns."),
]
comparison = pd.DataFrame(rows, columns=['member', 'columns', 'teammate_finding', 'rhea_issue', 'status', 'note'])
comparison.to_csv('../reports/team_combined_comparison.csv', index=False)
comparison

,member,columns,teammate_finding,rhea_issue,status,note
0,Emma,media_type,"80 nulls, 56 tied to the missing-metadata block","DQ-07, DQ-08",Agrees,Emma also suggests exploring dropping the block. Rhea keeps the rows with an 'unknown' category and a targeting_info...
1,Emma,profession,56 nulls,DQ-10,Agrees,
2,Emma,profession,"Numeric codes mixed with text, no codebook",DQ-09,Agrees,Both treat codes as categories and ask WebMD for the dictionary.
3,Emma,specialty,"'undefined' and nulls hide missing data, use npi_specialty",DQ-12,"Agrees, different fix",Emma removes specialty from the model. Rhea keeps it as 'unknown' plus a has_specialty flag.
4,Emma,specialty,11 numeric codes,DQ-13,"Agrees, different fix",Emma recodes them to NaN. Rhea keeps them as code_ labels.
5,Emma,campaign_start_date,"3 nulls, stored as text",DQ-14,"Agrees, different fix",Emma drops the 3 rows. Rhea imputes the company median date and flags them. Same 3 rows where total_spend is blank (...
6,Emma,campaign_end_date,"2099 placeholder on 414 rows, drop column",DQ-15,Agrees,
7,Emma,"profession, specialty",(not covered),DQ-11,Only in Rhea's report,36 rows pair psychologist with urology.
8,Abhiram,total_budget_spent,2 rows at $0 with real engagement,DQ-18,Agrees,Rhea also flags 5 more rows with budgets far below cpe x engagements (7 in total).
9,Abhiram,engagement_rate,Derived and rounded to 4 decimals,DQ-20,Agrees,"0 mismatches after rounding, so documenting the rounding is enough."


In [13]:
comparison.groupby(['member', 'status'], sort=False).size().unstack(fill_value=0)

status,Agrees,"Agrees, different fix",Only in Rhea's report,Adds detail,Rhea only
member,,,,,
Emma,4,3,1,0,0
Abhiram,4,0,1,0,0
Bakari,1,0,1,0,0
Timothy,2,3,1,0,0
Nida,1,0,1,1,0
Rhea,0,0,0,0,1


---
## 8. Decisions for the team

| # | Question | Options on the table |
|---|---|---|
| 1 | The 3 rows with no start date | Drop them (Emma) or impute and flag (Rhea). They are already out of spend training. |
| 2 | specialty column | Remove it (Emma) or keep as 'unknown' plus has_specialty (Rhea). Both use npi_specialty as the main feature. |
| 3 | Outliers in post_avg_NRx and pre_NRx_avg | Cap at the IQR limit (Timothy) or keep real values (Rhea). Neither is a model input for total_spend yet. |
| 4 | NRx_lift rounding | Recalculate (Timothy, and then recalculate NRx_lift_pct too) or keep as provided (Rhea). |
| 5 | Leakage columns | Agree as a team to keep post-campaign and spend-derived columns out of the feature sets (DQ-19, DQ-22, DQ-23, DQ-35). |

## 9. Getting every notebook to run from the repo

A few notebooks load the data from a path that only works on the author's machine. Changing the load line to `pd.read_excel('../data/BreakThroughTech_training_data_1.xlsx')` and running from the `notebooks` folder fixes it.

| Notebook | What to change |
|---|---|
| `Milestone1.ipynb` (Bakari) | Uses the Google Colab path `/content/...`. Switch to the repo path. |
| `BTT.ipynb` (Nida) | Loads from the working folder and uses `micropip` (JupyterLite). Switch to the repo path and drop the micropip cell. The first cell still shows an openpyxl error in its saved output. |
| `TimothyHsu_Data_Quality_Report.ipynb` (Timothy) | Loads from the working folder. Switch to the repo path. |
| `data_quality_report_variables_E-H.ipynb` (Emma) | Path is correct. The first cell's saved output shows a missing pandas error, so a clean rerun would tidy it up. |
| `data_quality_report_variables_J-N.ipynb` (Abhiram) | Path is correct. Uses matplotlib, which is not in `requirements.txt` yet. |